In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Appsb\\Desktop\\Text-Summarizer\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\Appsb\\Desktop\\Text-Summarizer'

In [5]:
#entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class modelTrainerConfig:
    rootDir: Path
    dataPath: Path
    modelCkpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int
    size: int

In [6]:
from textSummarizer.utils.common import read_yaml,createDir
from textSummarizer.constants import *

In [7]:
from logging import config

class configurationManager:
    def __init__(
        self,
        config_FilePath = configFilePath,
        params_FilePath = paramsFilePath):
        
        self.config = read_yaml(config_FilePath)
        self.params = read_yaml(params_FilePath)
        
        createDir([self.config.artifacts_root])
    
    def getModelTrainerConfig(self) -> modelTrainerConfig:
        config = self.config.modelTrainer
        params = self.params.trainingArguments
        
        createDir([config.rootDir])
        
        model_trainer_config = modelTrainerConfig(
            rootDir=config.rootDir,
            dataPath=config.dataPath,
            modelCkpt=config.modelCkpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            per_device_eval_batch_size=params.per_device_eval_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy= params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps,
            size= params.size,
        )
        return model_trainer_config

In [8]:
from transformers import TrainingArguments,Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from datasets import load_dataset,load_from_disk
import torch

c:\Users\Appsb\Desktop\Text-Summarizer\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-03-14 19:50:21,771: WARNING: module_wrapper: From c:\Users\Appsb\Desktop\Text-Summarizer\.conda\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]
[2025-03-14 19:50:22,975: INFO: config: PyTorch version 2.6.0 available.]
[2025-03-14 19:50:22,993: INFO: config: TensorFlow version 2.19.0 available.]


In [ ]:
class modelTrainer:
    def __init__(self,config:modelTrainerConfig):
        self.config = config



    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.modelCkpt)
        modelPagasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.modelCkpt).to(device)
        seq2seqDataCollator = DataCollatorForSeq2Seq(tokenizer,modelPagasus)
        
        dataset_samsum_pt = load_from_disk(self.config.dataPath)
        trainer_args = TrainingArguments(
            output_dir = str(self.config.rootDir), 
            num_train_epochs=self.config.num_train_epochs, 
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size, 
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay, 
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy, 
            eval_steps=self.config.eval_steps, save_steps=1e6,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        ) 

        trainer = Trainer(
            model=modelPagasus, 
            args=trainer_args,
            processing_class=tokenizer, 
            data_collator=seq2seqDataCollator,
            train_dataset=dataset_samsum_pt["train"], 
            eval_dataset=dataset_samsum_pt["validation"]
        )
        
        trainer.train()

        ## Save model
        modelPagasus.save_pretrained(os.path.join(self.config.rootDir,"pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.rootDir,"tokenizer"))

In [10]:
try:
    config = configurationManager()
    model_trainer_config = config.getModelTrainerConfig()
    model_trainer_config = modelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2025-03-14 17:37:19,169: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-03-14 17:37:19,179: INFO: common: yaml file: params.yaml loaded successfully]
[2025-03-14 17:37:19,184: INFO: common: Created director at: artifacts]
[2025-03-14 17:37:19,188: INFO: common: Created director at: artifacts/model_trainer]


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Appsb\Desktop\Text-Summarizer\.conda\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


: 